# Mixed Continuous-Discrete Bayesian Optimization

This tutorial demonstrates single-objective Bayesian optimization in Xopt for a mixed search space containing continuous and discrete variables.

We use a synthetic objective where a sinusoidal continuous landscape changes with a discrete phase value. This makes the optimizer balance:

- global exploration over discrete choices
- local optimization in a continuous coordinate

In [ ]:
import math
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

from xopt import Evaluator, Xopt
from xopt.generators.bayesian import ExpectedImprovementGenerator
from xopt.vocs import VOCS

warnings.filterwarnings("ignore")

SMOKE_TEST = os.environ.get("SMOKE_TEST")

# Keep smoke runs fast for docs validation while preserving behavior.
N_INITIAL = 3 if SMOKE_TEST else 5
N_STEPS = 2 if SMOKE_TEST else 25
N_RESTARTS = 1 if SMOKE_TEST else 8
N_MC_SAMPLES = 8 if SMOKE_TEST else 128
N_GRID = 30 if SMOKE_TEST else 100
SEED = 123

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.set_num_threads(1)

## Define a mixed search space

The variable `phase` is discrete with values `{0.0, 0.8, 1.6, 2.4}`.

The objective is

$$
f(x_1, \phi) = (\cos(x_1 + \phi) - 0.25)^2 + 0.1\sin(3x_1 + \phi) + 0.05\phi,
$$

where $\phi$ is optimized directly as a discrete phase offset.

This keeps the mixed optimization example simple while still showing joint continuous-discrete behavior.

In [ ]:
PHASE_VALUES = [0.0, 0.8, 1.6, 2.4]


def evaluate_mixed_sinusoid(inputs):
    x1 = float(inputs["x1"])
    phase = float(inputs["phase"])

    value = (
        (math.cos(x1 + phase) - 0.25) ** 2
        + 0.1 * math.sin(3.0 * x1 + phase)
        + 0.05 * phase
    )
    return {"f": value}


vocs = VOCS(
    variables={
        "x1": [0.0, 2.0 * math.pi],
        "phase": set(PHASE_VALUES),
    },
    objectives={"f": "MINIMIZE"},
)

vocs

## Create Xopt objects

We use `ExpectedImprovementGenerator` and standard GP modeling options, with smoke-test-aware optimizer settings for quick docs checks.

In [ ]:
evaluator = Evaluator(function=evaluate_mixed_sinusoid)
generator = ExpectedImprovementGenerator(vocs=vocs)
generator.gp_constructor.use_low_noise_prior = True
generator.numerical_optimizer.n_restarts = N_RESTARTS
generator.n_monte_carlo_samples = N_MC_SAMPLES

X = Xopt(evaluator=evaluator, generator=generator)

X.random_evaluate(N_INITIAL)
for _ in range(N_STEPS):
    X.step()

X.data

## Optimization diagnostics

We check three things:

1. best-so-far objective over evaluations
2. frequency of sampled phase values (rounded for display)
3. whether sampled phases exactly match the configured discrete set

In [ ]:
data = X.data.copy()
data["eval_index"] = np.arange(len(data))
data["best_so_far"] = data["f"].cummin()

data["phase_raw"] = data["phase"].astype(float)
data["phase_display"] = data["phase_raw"].round(3)

allowed_phase = set(float(v) for v in PHASE_VALUES)
phase_is_configured = data["phase_raw"].apply(
    lambda value: any(np.isclose(value, v, atol=1e-8) for v in PHASE_VALUES)
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(data["eval_index"], data["f"], "o-", alpha=0.5, label="objective")
ax[0].plot(data["eval_index"], data["best_so_far"], "k-", lw=2, label="best so far")
ax[0].set_xlabel("evaluation")
ax[0].set_ylabel("f")
ax[0].legend()

count_series = data["phase_display"].value_counts().sort_index()
ax[1].bar(count_series.index.astype(str), count_series.values, color="C1")
ax[1].set_xlabel("sampled phase (rounded)")
ax[1].set_ylabel("sample count")

fig.tight_layout()

print(f"Configured phase set: {sorted(allowed_phase)}")
print(
    f"Exact configured phase matches: {int(phase_is_configured.sum())}/{len(phase_is_configured)}"
)
if not phase_is_configured.all():
    print("Some sampled phase values are not exact configured values.")

## Model and acquisition slices by discrete phase offset

To isolate the effect of the discrete offset, we visualize the model and acquisition over `(x1, phase)`.

In [ ]:
X.generator.visualize_model(variable_names=["x1", "phase"])

## Best candidate and summary

The best observed point combines a continuous location `x1` and a discrete phase offset.

In this example, Bayesian optimization jointly learns:

- which phase offset gives the best regime
- where to place continuous evaluations inside that regime

In [ ]:
best_row = data.loc[data["f"].idxmin(), ["x1", "phase", "f"]]
summary = (
    data.groupby("phase_display", as_index=False)["f"]
    .min()
    .rename(columns={"f": "best_f_at_phase"})
)

print("Best observed candidate:")
display(best_row.to_frame().T)

print("Best objective reached at each sampled phase (rounded):")
display(summary.sort_values("best_f_at_phase"))